# 02. Feature Engineering

Objetivo:
- generar features pre-partido sin fuga de informacion
- construir ratings dinamicos y forma reciente
- exportar un dataset maestro para modelado probabilistico
- exportar una version final mas limpia para referencia

In [32]:
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

cleanPath = Path("../data/clean/clean_matches.csv")
modelingBasePath = Path("../data/intermediate/modeling_base_dataset.csv")
finalModelPath = Path("../data/finalModelDf/finalModelDFmodeling_dataset.csv")

print("Clean path:", cleanPath)
print("Modeling base path:", modelingBasePath)
print("Final model path:", finalModelPath)

Clean path: ..\data\clean\clean_matches.csv
Modeling base path: ..\data\intermediate\modeling_base_dataset.csv
Final model path: ..\data\finalModelDf\finalModelDFmodeling_dataset.csv


## 1. Carga del dataset limpio

In [33]:
df = pd.read_csv(cleanPath)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date", kind="mergesort").reset_index(drop=True)

print("Shape clean matches:", df.shape)
df.head()

Shape clean matches: (25013, 11)


,date,homeTeam,awayTeam,homeScore,awayScore,tournament,city,country,neutral,matchResult,totalGoals
0,2000-01-04,Egypt,Togo,2,1,Friendly,Aswan,Egypt,False,win,3
1,2000-01-07,Tunisia,Togo,7,0,Friendly,Tunis,Tunisia,False,win,7
2,2000-01-08,Trinidad and Tobago,Canada,0,0,Friendly,Port of Spain,Trinidad and Tobago,False,draw,0
3,2000-01-09,Burkina Faso,Gabon,1,1,Friendly,Ouagadougou,Burkina Faso,False,draw,2
4,2000-01-09,Guatemala,Armenia,1,1,Friendly,Los Angeles,United States,True,draw,2


## 2. Validacion previa

In [34]:
requiredColumns = [
    "date",
    "homeTeam",
    "awayTeam",
    "homeScore",
    "awayScore",
    "tournament",
    "city",
    "country",
    "neutral",
    "matchResult",
    "totalGoals",
]

missingColumns = [col for col in requiredColumns if col not in df.columns]
assert not missingColumns, f"Faltan columnas requeridas: {missingColumns}"
assert df[requiredColumns].isnull().sum().sum() == 0, "Hay nulos en columnas requeridas"
assert (df[["homeScore", "awayScore"]] < 0).sum().sum() == 0, "Hay marcadores negativos"
assert df.duplicated(subset=["date", "homeTeam", "awayTeam"]).sum() == 0, "Hay llaves duplicadas de partido"

print("Rango de fechas:", df["date"].min(), "->", df["date"].max())
print("Distribucion del target:")
print(df["matchResult"].value_counts(normalize=True).round(4))

Rango de fechas: 2000-01-04 00:00:00 -> 2026-01-26 00:00:00
Distribucion del target:
matchResult
win     0.4811
loss    0.2859
draw    0.2329
Name: proportion, dtype: float64


## 3. Funciones auxiliares

In [35]:
def safeMean(values):
    if len(values) == 0:
        return np.nan
    return float(np.mean(values))


def safeRate(numerator, denominator):
    if denominator == 0:
        return np.nan
    return float(numerator / denominator)


def getResultPoints(homeScore, awayScore):
    if homeScore > awayScore:
        return 3, 0
    if homeScore < awayScore:
        return 0, 3
    return 1, 1


def categorizeTournament(tournament):
    tournamentText = str(tournament).strip().lower()

    if any(keyword in tournamentText for keyword in ["world cup", "fifa world cup", "mundial"]):
        return "worldCup"
    if any(keyword in tournamentText for keyword in ["qualification", "qualifier", "qualifiers", "eliminatoria", "eliminatorias"]):
        return "qualification"
    if any(keyword in tournamentText for keyword in ["friendly", "amistoso", "amistosos"]):
        return "friendly"

    continentalKeywords = [
        "copa america",
        "conmebol",
        "euro",
        "european championship",
        "african cup",
        "afcon",
        "asian cup",
        "gold cup",
        "concacaf",
        "nations league",
        "uefa nations",
        "oceania",
        "ofc",
    ]
    if any(keyword in tournamentText for keyword in continentalKeywords):
        return "continental"

    return "other"


def createTeamState():
    return {
        "elo": 1500.0,
        "matchesPlayed": 0,
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "goalsFor": 0,
        "goalsAgainst": 0,
        "lastMatchDate": None,
        "last5Points": deque(maxlen=5),
        "last5GoalsFor": deque(maxlen=5),
        "last5GoalsAgainst": deque(maxlen=5),
        "last10Points": deque(maxlen=10),
        "last10GoalsFor": deque(maxlen=10),
        "last10GoalsAgainst": deque(maxlen=10),
    }


def createPairState():
    return {
        "matchesPlayed": 0,
        "firstTeamWins": 0,
        "secondTeamWins": 0,
        "draws": 0,
    }


def getPairKey(teamA, teamB):
    return tuple(sorted([teamA, teamB]))


baseK = 20
homeAdvantage = 65


def expectedScore(ratingA, ratingB):
    return 1 / (1 + 10 ** ((ratingB - ratingA) / 400))


def updateElo(homeElo, awayElo, homeScore, awayScore, neutral=False):
    adjustedHomeElo = homeElo if neutral else homeElo + homeAdvantage
    expectedHome = expectedScore(adjustedHomeElo, awayElo)

    if homeScore > awayScore:
        actualHome = 1.0
    elif homeScore < awayScore:
        actualHome = 0.0
    else:
        actualHome = 0.5

    newHomeElo = homeElo + baseK * (actualHome - expectedHome)
    newAwayElo = awayElo + baseK * ((1 - actualHome) - (1 - expectedHome))
    return newHomeElo, newAwayElo, expectedHome

## 4. Generacion del dataset de modelado

In [36]:
teamState = defaultdict(createTeamState)
pairState = defaultdict(createPairState)
featureRows = []

for row in df.itertuples(index=False):
    matchDate = row.date
    homeTeam = row.homeTeam
    awayTeam = row.awayTeam
    homeScore = int(row.homeScore)
    awayScore = int(row.awayScore)
    tournament = row.tournament
    neutralFlag = int(bool(row.neutral))

    homeState = teamState[homeTeam]
    awayState = teamState[awayTeam]

    homeElo = homeState["elo"]
    awayElo = awayState["elo"]
    eloDiff = homeElo - awayElo
    absEloDiff = abs(eloDiff)
    isCloseMatch = 1 if absEloDiff < 50 else 0
    isVeryCloseMatch = 1 if absEloDiff < 25 else 0
    eloExpectedHomeWin = expectedScore(homeElo if neutralFlag else homeElo + homeAdvantage, awayElo)

    homeMatchesPlayed = homeState["matchesPlayed"]
    awayMatchesPlayed = awayState["matchesPlayed"]

    homeWinRate = safeRate(homeState["wins"], homeMatchesPlayed)
    awayWinRate = safeRate(awayState["wins"], awayMatchesPlayed)
    homeDrawRate = safeRate(homeState["draws"], homeMatchesPlayed)
    awayDrawRate = safeRate(awayState["draws"], awayMatchesPlayed)
    homeLossRate = safeRate(homeState["losses"], homeMatchesPlayed)
    awayLossRate = safeRate(awayState["losses"], awayMatchesPlayed)

    homeGoalsForAvg = safeRate(homeState["goalsFor"], homeMatchesPlayed)
    awayGoalsForAvg = safeRate(awayState["goalsFor"], awayMatchesPlayed)
    totalGoalsAvg = (
        homeGoalsForAvg + awayGoalsForAvg
        if pd.notna(homeGoalsForAvg) and pd.notna(awayGoalsForAvg)
        else np.nan
    )

    lowScoringSignal = 1 if (pd.notna(totalGoalsAvg) and totalGoalsAvg < 2.5) else 0
    lowTotalGoalsStrong = 1 if (pd.notna(totalGoalsAvg) and totalGoalsAvg < 2.2) else 0

    drawTendency = (
        absEloDiff * totalGoalsAvg
        if pd.notna(totalGoalsAvg)
        else np.nan
    )


    homeGoalsAgainstAvg = safeRate(homeState["goalsAgainst"], homeMatchesPlayed)
    awayGoalsAgainstAvg = safeRate(awayState["goalsAgainst"], awayMatchesPlayed)
    homeGoalDiffAvg = safeRate(homeState["goalsFor"] - homeState["goalsAgainst"], homeMatchesPlayed)
    awayGoalDiffAvg = safeRate(awayState["goalsFor"] - awayState["goalsAgainst"], awayMatchesPlayed)

    homeLast5WinRate = safeMean([1 if points == 3 else 0 for points in homeState["last5Points"]])
    awayLast5WinRate = safeMean([1 if points == 3 else 0 for points in awayState["last5Points"]])
    homeLast5PointsAvg = safeMean(homeState["last5Points"])
    awayLast5PointsAvg = safeMean(awayState["last5Points"])
    homeLast5GoalsForAvg = safeMean(homeState["last5GoalsFor"])
    awayLast5GoalsForAvg = safeMean(awayState["last5GoalsFor"])
    homeLast5GoalsAgainstAvg = safeMean(homeState["last5GoalsAgainst"])
    awayLast5GoalsAgainstAvg = safeMean(awayState["last5GoalsAgainst"])
    homeLast5GoalDiffAvg = safeMean([gf - ga for gf, ga in zip(homeState["last5GoalsFor"], homeState["last5GoalsAgainst"])])
    awayLast5GoalDiffAvg = safeMean([gf - ga for gf, ga in zip(awayState["last5GoalsFor"], awayState["last5GoalsAgainst"])])

    homeLast10PointsAvg = safeMean(homeState["last10Points"])
    awayLast10PointsAvg = safeMean(awayState["last10Points"])
    homeLast10GoalsForAvg = safeMean(homeState["last10GoalsFor"])
    awayLast10GoalsForAvg = safeMean(awayState["last10GoalsFor"])
    homeLast10GoalsAgainstAvg = safeMean(homeState["last10GoalsAgainst"])
    awayLast10GoalsAgainstAvg = safeMean(awayState["last10GoalsAgainst"])

    homeDaysSinceLastMatch = (matchDate - homeState["lastMatchDate"]).days if homeState["lastMatchDate"] is not None else np.nan
    awayDaysSinceLastMatch = (matchDate - awayState["lastMatchDate"]).days if awayState["lastMatchDate"] is not None else np.nan

    pairKey = getPairKey(homeTeam, awayTeam)
    pairStats = pairState[pairKey]
    if pairStats["matchesPlayed"] > 0:
        if homeTeam == pairKey[0]:
            homeH2HWinRate = pairStats["firstTeamWins"] / pairStats["matchesPlayed"]
            awayH2HWinRate = pairStats["secondTeamWins"] / pairStats["matchesPlayed"]
        else:
            homeH2HWinRate = pairStats["secondTeamWins"] / pairStats["matchesPlayed"]
            awayH2HWinRate = pairStats["firstTeamWins"] / pairStats["matchesPlayed"]
        h2hDrawRate = pairStats["draws"] / pairStats["matchesPlayed"]
    else:
        homeH2HWinRate = np.nan
        awayH2HWinRate = np.nan
        h2hDrawRate = np.nan

    tournamentCategory = categorizeTournament(tournament)
    target = row.matchResult

    featureRows.append(
        {
            "date": matchDate,
            "homeTeam": homeTeam,
            "awayTeam": awayTeam,
            "tournament": tournament,
            "tournamentCategory": tournamentCategory,
            "neutral": neutralFlag,
            "matchMonth": matchDate.month,
            "matchYear": matchDate.year,
            "isCloseMatch": isCloseMatch,
            "isVeryCloseMatch": isVeryCloseMatch,
            "totalGoalsAvg": totalGoalsAvg,
            "lowScoringSignal": lowScoringSignal,
            "lowTotalGoalsStrong": lowTotalGoalsStrong,
            "drawTendency": drawTendency,
            "homeScore": homeScore,
            "awayScore": awayScore,
            "homeElo": homeElo,
            "awayElo": awayElo,
            "eloDiff": eloDiff,
            "absEloDiff": absEloDiff,
            "eloExpectedHomeWin": eloExpectedHomeWin,
            "homeMatchesPlayed": homeMatchesPlayed,
            "awayMatchesPlayed": awayMatchesPlayed,
            "matchesPlayedDiff": homeMatchesPlayed - awayMatchesPlayed,
            "homeWinRate": homeWinRate,
            "awayWinRate": awayWinRate,
            "winRateDiff": homeWinRate - awayWinRate if pd.notna(homeWinRate) and pd.notna(awayWinRate) else np.nan,
            "homeDrawRate": homeDrawRate,
            "awayDrawRate": awayDrawRate,
            "drawRateDiff": homeDrawRate - awayDrawRate if pd.notna(homeDrawRate) and pd.notna(awayDrawRate) else np.nan,
            "homeLossRate": homeLossRate,
            "awayLossRate": awayLossRate,
            "lossRateDiff": homeLossRate - awayLossRate if pd.notna(homeLossRate) and pd.notna(awayLossRate) else np.nan,
            "homeGoalsForAvg": homeGoalsForAvg,
            "awayGoalsForAvg": awayGoalsForAvg,
            "goalsForAvgDiff": homeGoalsForAvg - awayGoalsForAvg if pd.notna(homeGoalsForAvg) and pd.notna(awayGoalsForAvg) else np.nan,
            "homeGoalsAgainstAvg": homeGoalsAgainstAvg,
            "awayGoalsAgainstAvg": awayGoalsAgainstAvg,
            "goalsAgainstAvgDiff": homeGoalsAgainstAvg - awayGoalsAgainstAvg if pd.notna(homeGoalsAgainstAvg) and pd.notna(awayGoalsAgainstAvg) else np.nan,
            "homeGoalDiffAvg": homeGoalDiffAvg,
            "awayGoalDiffAvg": awayGoalDiffAvg,
            "goalDiffAvgDiff": homeGoalDiffAvg - awayGoalDiffAvg if pd.notna(homeGoalDiffAvg) and pd.notna(awayGoalDiffAvg) else np.nan,
            "homeLast5WinRate": homeLast5WinRate,
            "awayLast5WinRate": awayLast5WinRate,
            "last5WinRateDiff": homeLast5WinRate - awayLast5WinRate if pd.notna(homeLast5WinRate) and pd.notna(awayLast5WinRate) else np.nan,
            "homeLast5PointsAvg": homeLast5PointsAvg,
            "awayLast5PointsAvg": awayLast5PointsAvg,
            "last5PointsDiff": homeLast5PointsAvg - awayLast5PointsAvg if pd.notna(homeLast5PointsAvg) and pd.notna(awayLast5PointsAvg) else np.nan,
            "homeLast5GoalsForAvg": homeLast5GoalsForAvg,
            "awayLast5GoalsForAvg": awayLast5GoalsForAvg,
            "last5GoalsForDiff": homeLast5GoalsForAvg - awayLast5GoalsForAvg if pd.notna(homeLast5GoalsForAvg) and pd.notna(awayLast5GoalsForAvg) else np.nan,
            "homeLast5GoalsAgainstAvg": homeLast5GoalsAgainstAvg,
            "awayLast5GoalsAgainstAvg": awayLast5GoalsAgainstAvg,
            "last5GoalsAgainstDiff": homeLast5GoalsAgainstAvg - awayLast5GoalsAgainstAvg if pd.notna(homeLast5GoalsAgainstAvg) and pd.notna(awayLast5GoalsAgainstAvg) else np.nan,
            "homeLast5GoalDiffAvg": homeLast5GoalDiffAvg,
            "awayLast5GoalDiffAvg": awayLast5GoalDiffAvg,
            "last5GoalDiffDiff": homeLast5GoalDiffAvg - awayLast5GoalDiffAvg if pd.notna(homeLast5GoalDiffAvg) and pd.notna(awayLast5GoalDiffAvg) else np.nan,
            "homeLast10PointsAvg": homeLast10PointsAvg,
            "awayLast10PointsAvg": awayLast10PointsAvg,
            "last10PointsDiff": homeLast10PointsAvg - awayLast10PointsAvg if pd.notna(homeLast10PointsAvg) and pd.notna(awayLast10PointsAvg) else np.nan,
            "homeLast10GoalsForAvg": homeLast10GoalsForAvg,
            "awayLast10GoalsForAvg": awayLast10GoalsForAvg,
            "last10GoalsForDiff": homeLast10GoalsForAvg - awayLast10GoalsForAvg if pd.notna(homeLast10GoalsForAvg) and pd.notna(awayLast10GoalsForAvg) else np.nan,
            "homeLast10GoalsAgainstAvg": homeLast10GoalsAgainstAvg,
            "awayLast10GoalsAgainstAvg": awayLast10GoalsAgainstAvg,
            "last10GoalsAgainstDiff": homeLast10GoalsAgainstAvg - awayLast10GoalsAgainstAvg if pd.notna(homeLast10GoalsAgainstAvg) and pd.notna(awayLast10GoalsAgainstAvg) else np.nan,
            "homeDaysSinceLastMatch": homeDaysSinceLastMatch,
            "awayDaysSinceLastMatch": awayDaysSinceLastMatch,
            "daysSinceLastMatchDiff": homeDaysSinceLastMatch - awayDaysSinceLastMatch if pd.notna(homeDaysSinceLastMatch) and pd.notna(awayDaysSinceLastMatch) else np.nan,
            "homeH2HWinRate": homeH2HWinRate,
            "awayH2HWinRate": awayH2HWinRate,
            "h2hDrawRate": h2hDrawRate,
            "target": target,
        }
    )

    homePoints, awayPoints = getResultPoints(homeScore, awayScore)

    homeState["matchesPlayed"] += 1
    awayState["matchesPlayed"] += 1
    homeState["wins"] += int(homeScore > awayScore)
    homeState["draws"] += int(homeScore == awayScore)
    homeState["losses"] += int(homeScore < awayScore)
    awayState["wins"] += int(awayScore > homeScore)
    awayState["draws"] += int(homeScore == awayScore)
    awayState["losses"] += int(awayScore < homeScore)

    homeState["goalsFor"] += homeScore
    homeState["goalsAgainst"] += awayScore
    awayState["goalsFor"] += awayScore
    awayState["goalsAgainst"] += homeScore

    homeState["last5Points"].append(homePoints)
    awayState["last5Points"].append(awayPoints)
    homeState["last5GoalsFor"].append(homeScore)
    homeState["last5GoalsAgainst"].append(awayScore)
    awayState["last5GoalsFor"].append(awayScore)
    awayState["last5GoalsAgainst"].append(homeScore)

    homeState["last10Points"].append(homePoints)
    awayState["last10Points"].append(awayPoints)
    homeState["last10GoalsFor"].append(homeScore)
    homeState["last10GoalsAgainst"].append(awayScore)
    awayState["last10GoalsFor"].append(awayScore)
    awayState["last10GoalsAgainst"].append(homeScore)

    homeState["lastMatchDate"] = matchDate
    awayState["lastMatchDate"] = matchDate

    newHomeElo, newAwayElo, _ = updateElo(homeElo, awayElo, homeScore, awayScore, neutral=bool(neutralFlag))
    homeState["elo"] = newHomeElo
    awayState["elo"] = newAwayElo

    pairStats["matchesPlayed"] += 1
    if homeScore > awayScore:
        if homeTeam == pairKey[0]:
            pairStats["firstTeamWins"] += 1
        else:
            pairStats["secondTeamWins"] += 1
    elif homeScore < awayScore:
        if awayTeam == pairKey[0]:
            pairStats["firstTeamWins"] += 1
        else:
            pairStats["secondTeamWins"] += 1
    else:
        pairStats["draws"] += 1

modelDf = pd.DataFrame(featureRows)
modelDf = modelDf.sort_values("date", kind="mergesort").reset_index(drop=True)

print("Shape modelDf:", modelDf.shape)
modelDf.head()

Shape modelDf: (25013, 73)


,date,homeTeam,awayTeam,tournament,tournamentCategory,neutral,matchMonth,matchYear,isCloseMatch,isVeryCloseMatch,totalGoalsAvg,lowScoringSignal,lowTotalGoalsStrong,drawTendency,homeScore,awayScore,homeElo,awayElo,eloDiff,absEloDiff,eloExpectedHomeWin,homeMatchesPlayed,awayMatchesPlayed,matchesPlayedDiff,homeWinRate,awayWinRate,winRateDiff,homeDrawRate,awayDrawRate,drawRateDiff,homeLossRate,awayLossRate,lossRateDiff,homeGoalsForAvg,awayGoalsForAvg,goalsForAvgDiff,homeGoalsAgainstAvg,awayGoalsAgainstAvg,goalsAgainstAvgDiff,homeGoalDiffAvg,awayGoalDiffAvg,goalDiffAvgDiff,homeLast5WinRate,awayLast5WinRate,last5WinRateDiff,homeLast5PointsAvg,awayLast5PointsAvg,last5PointsDiff,homeLast5GoalsForAvg,awayLast5GoalsForAvg,last5GoalsForDiff,homeLast5GoalsAgainstAvg,awayLast5GoalsAgainstAvg,last5GoalsAgainstDiff,homeLast5GoalDiffAvg,awayLast5GoalDiffAvg,last5GoalDiffDiff,homeLast10PointsAvg,awayLast10PointsAvg,last10PointsDiff,homeLast10GoalsForAvg,awayLast10GoalsForAvg,last10GoalsForDiff,homeLast10GoalsAgainstAvg,awayLast10GoalsAgainstAvg,last10GoalsAgainstDiff,homeDaysSinceLastMatch,awayDaysSinceLastMatch,daysSinceLastMatchDiff,homeH2HWinRate,awayH2HWinRate,h2hDrawRate,target
0,2000-01-04,Egypt,Togo,Friendly,friendly,0,1,2000,1,1,NaN,0,0,NaN,2,1,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,win
1,2000-01-07,Tunisia,Togo,Friendly,friendly,0,1,2000,1,1,NaN,0,0,NaN,7,0,1500.0,1491.849325,8.150675,8.150675,0.603744,0,1,-1,NaN,0.0,NaN,NaN,0.0,NaN,NaN,1.0,NaN,NaN,1.0,NaN,NaN,2.0,NaN,NaN,-1.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,1.0,NaN,NaN,2.0,NaN,NaN,-1.0,NaN,NaN,0.0,NaN,NaN,1.0,NaN,NaN,2.0,NaN,NaN,3.0,NaN,NaN,NaN,NaN,win
2,2000-01-08,Trinidad and Tobago,Canada,Friendly,friendly,0,1,2000,1,1,NaN,0,0,NaN,0,0,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,draw
3,2000-01-09,Burkina Faso,Gabon,Friendly,friendly,0,1,2000,1,1,NaN,0,0,NaN,1,1,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,draw
4,2000-01-09,Guatemala,Armenia,Friendly,friendly,1,1,2000,1,1,NaN,0,0,NaN,1,1,1500.0,1500.000000,0.000000,0.000000,0.500000,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,draw


In [37]:
nullSummary = modelDf.isnull().mean().sort_values(ascending=False)
nullSummary[nullSummary > 0].head(20)

awayH2HWinRate            0.253668
h2hDrawRate               0.253668
homeH2HWinRate            0.253668
goalsAgainstAvgDiff       0.010515
goalsForAvgDiff           0.010515
totalGoalsAvg             0.010515
drawRateDiff              0.010515
drawTendency              0.010515
last5GoalsForDiff         0.010515
last10GoalsAgainstDiff    0.010515
last10GoalsForDiff        0.010515
daysSinceLastMatchDiff    0.010515
last5GoalDiffDiff         0.010515
goalDiffAvgDiff           0.010515
lossRateDiff              0.010515
winRateDiff               0.010515
last10PointsDiff          0.010515
last5WinRateDiff          0.010515
last5PointsDiff           0.010515
last5GoalsAgainstDiff     0.010515
dtype: float64

## 5. Imputacion controlada y target encoding

In [38]:
numericColumns = modelDf.select_dtypes(include=[np.number]).columns.tolist()
for col in numericColumns:
    if modelDf[col].isnull().any():
        modelDf[col] = modelDf[col].fillna(modelDf[col].median())

targetMapping = {"loss": 0, "draw": 1, "win": 2}
modelDf["targetEncoded"] = modelDf["target"].map(targetMapping)

modelDf = pd.get_dummies(
    modelDf,
    columns=["tournamentCategory"],
    prefix="tournamentCat",
    dtype=int,
)

remainingNulls = modelDf.isnull().sum().sum()
assert remainingNulls == 0, f"Quedaron nulos en modelDf: {remainingNulls}"

idColumns = [
    "date",
    "homeTeam",
    "awayTeam",
    "tournament",
    "neutral",
    "matchMonth",
    "matchYear",
    "homeScore",
    "awayScore",
    "target",
    "targetEncoded",
]
otherColumns = [col for col in modelDf.columns if col not in idColumns]
modelDf = modelDf[idColumns + otherColumns]

print("Shape modelDf luego de imputacion y encoding:", modelDf.shape)
modelDf.head()

Shape modelDf luego de imputacion y encoding: (25013, 78)


,date,homeTeam,awayTeam,tournament,neutral,matchMonth,matchYear,homeScore,awayScore,target,targetEncoded,isCloseMatch,isVeryCloseMatch,totalGoalsAvg,lowScoringSignal,lowTotalGoalsStrong,drawTendency,homeElo,awayElo,eloDiff,absEloDiff,eloExpectedHomeWin,homeMatchesPlayed,awayMatchesPlayed,matchesPlayedDiff,homeWinRate,awayWinRate,winRateDiff,homeDrawRate,awayDrawRate,drawRateDiff,homeLossRate,awayLossRate,lossRateDiff,homeGoalsForAvg,awayGoalsForAvg,goalsForAvgDiff,homeGoalsAgainstAvg,awayGoalsAgainstAvg,goalsAgainstAvgDiff,homeGoalDiffAvg,awayGoalDiffAvg,goalDiffAvgDiff,homeLast5WinRate,awayLast5WinRate,last5WinRateDiff,homeLast5PointsAvg,awayLast5PointsAvg,last5PointsDiff,homeLast5GoalsForAvg,awayLast5GoalsForAvg,last5GoalsForDiff,homeLast5GoalsAgainstAvg,awayLast5GoalsAgainstAvg,last5GoalsAgainstDiff,homeLast5GoalDiffAvg,awayLast5GoalDiffAvg,last5GoalDiffDiff,homeLast10PointsAvg,awayLast10PointsAvg,last10PointsDiff,homeLast10GoalsForAvg,awayLast10GoalsForAvg,last10GoalsForDiff,homeLast10GoalsAgainstAvg,awayLast10GoalsAgainstAvg,last10GoalsAgainstDiff,homeDaysSinceLastMatch,awayDaysSinceLastMatch,daysSinceLastMatchDiff,homeH2HWinRate,awayH2HWinRate,h2hDrawRate,tournamentCat_continental,tournamentCat_friendly,tournamentCat_other,tournamentCat_qualification,tournamentCat_worldCup
0,2000-01-04,Egypt,Togo,Friendly,0,1,2000,2,1,win,2,1,1,2.744185,0,0,215.335666,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
1,2000-01-07,Tunisia,Togo,Friendly,0,1,2000,7,0,win,2,1,1,2.744185,0,0,215.335666,1500.0,1491.849325,8.150675,8.150675,0.603744,0,1,-1,0.391509,0.000000,0.015171,0.23913,0.000000,0.0,0.352941,1.000000,-0.017047,1.392857,1.000000,0.043808,1.225806,2.000000,-0.047008,0.185185,-1.000000,0.09632,0.4,0.0,0.0,1.4,0.0,0.0,1.2,1.0,0.0,1.2,2.0,0.0,0.0,-1.0,0.0,1.4,0.0,0.0,1.3,1.0,0.0,1.2,2.0,0.0,12.0,3.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
2,2000-01-08,Trinidad and Tobago,Canada,Friendly,0,1,2000,0,0,draw,1,1,1,2.744185,0,0,215.335666,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
3,2000-01-09,Burkina Faso,Gabon,Friendly,0,1,2000,1,1,draw,1,1,1,2.744185,0,0,215.335666,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
4,2000-01-09,Guatemala,Armenia,Friendly,1,1,2000,1,1,draw,1,1,1,2.744185,0,0,215.335666,1500.0,1500.000000,0.000000,0.000000,0.500000,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0


## 6. Exportacion de datasets

In [39]:
finalModelDf = modelDf.drop(
    columns=[
        "homeTeam",
        "awayTeam",
        "tournament",
        "homeScore",
        "awayScore",
    ]
).copy()

featureColumns = [
    col
    for col in modelDf.columns
    if col
    not in [
        "date",
        "homeTeam",
        "awayTeam",
        "tournament",
        "homeScore",
        "awayScore",
        "target",
        "targetEncoded",
    ]
]

modelingBasePath.parent.mkdir(parents=True, exist_ok=True)
finalModelPath.parent.mkdir(parents=True, exist_ok=True)

modelDf.to_csv(modelingBasePath, index=False)
finalModelDf.to_csv(finalModelPath, index=False)

print("Numero de features utiles:", len(featureColumns))
print("Shape modeling base:", modelDf.shape)
print("Shape final model df:", finalModelDf.shape)
print("Exportaciones completadas.")

Numero de features utiles: 70
Shape modeling base: (25013, 78)
Shape final model df: (25013, 73)
Exportaciones completadas.


In [40]:
reloadedBaseDf = pd.read_csv(modelingBasePath, nrows=3)
reloadedFinalDf = pd.read_csv(finalModelPath, nrows=3)

print("Columnas modeling base:", len(reloadedBaseDf.columns))
print("Columnas final model df:", len(reloadedFinalDf.columns))
reloadedBaseDf.head()

Columnas modeling base: 78
Columnas final model df: 73


,date,homeTeam,awayTeam,tournament,neutral,matchMonth,matchYear,homeScore,awayScore,target,targetEncoded,isCloseMatch,isVeryCloseMatch,totalGoalsAvg,lowScoringSignal,lowTotalGoalsStrong,drawTendency,homeElo,awayElo,eloDiff,absEloDiff,eloExpectedHomeWin,homeMatchesPlayed,awayMatchesPlayed,matchesPlayedDiff,homeWinRate,awayWinRate,winRateDiff,homeDrawRate,awayDrawRate,drawRateDiff,homeLossRate,awayLossRate,lossRateDiff,homeGoalsForAvg,awayGoalsForAvg,goalsForAvgDiff,homeGoalsAgainstAvg,awayGoalsAgainstAvg,goalsAgainstAvgDiff,homeGoalDiffAvg,awayGoalDiffAvg,goalDiffAvgDiff,homeLast5WinRate,awayLast5WinRate,last5WinRateDiff,homeLast5PointsAvg,awayLast5PointsAvg,last5PointsDiff,homeLast5GoalsForAvg,awayLast5GoalsForAvg,last5GoalsForDiff,homeLast5GoalsAgainstAvg,awayLast5GoalsAgainstAvg,last5GoalsAgainstDiff,homeLast5GoalDiffAvg,awayLast5GoalDiffAvg,last5GoalDiffDiff,homeLast10PointsAvg,awayLast10PointsAvg,last10PointsDiff,homeLast10GoalsForAvg,awayLast10GoalsForAvg,last10GoalsForDiff,homeLast10GoalsAgainstAvg,awayLast10GoalsAgainstAvg,last10GoalsAgainstDiff,homeDaysSinceLastMatch,awayDaysSinceLastMatch,daysSinceLastMatchDiff,homeH2HWinRate,awayH2HWinRate,h2hDrawRate,tournamentCat_continental,tournamentCat_friendly,tournamentCat_other,tournamentCat_qualification,tournamentCat_worldCup
0,2000-01-04,Egypt,Togo,Friendly,0,1,2000,2,1,win,2,1,1,2.744185,0,0,215.335666,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
1,2000-01-07,Tunisia,Togo,Friendly,0,1,2000,7,0,win,2,1,1,2.744185,0,0,215.335666,1500.0,1491.849325,8.150675,8.150675,0.603744,0,1,-1,0.391509,0.000000,0.015171,0.23913,0.000000,0.0,0.352941,1.000000,-0.017047,1.392857,1.000000,0.043808,1.225806,2.000000,-0.047008,0.185185,-1.000000,0.09632,0.4,0.0,0.0,1.4,0.0,0.0,1.2,1.0,0.0,1.2,2.0,0.0,0.0,-1.0,0.0,1.4,0.0,0.0,1.3,1.0,0.0,1.2,2.0,0.0,12.0,3.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
2,2000-01-08,Trinidad and Tobago,Canada,Friendly,0,1,2000,0,0,draw,1,1,1,2.744185,0,0,215.335666,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
